# Week 95, Day 2: Writing the fix

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w095d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('wraptext.py', 'w', encoding='utf-8') as f:
    f.write('"""Shorten text to fit a width.\n\n>>> shorten("Hello world, nice day", 18)\n\'Hello world, [...]\'\n"""\n\n\ndef shorten(text, width, placeholder=" [...]"):\n    """Collapse whitespace and cut text to fit width, ending with placeholder if anything was cut."""\n    words = text.split()\n    joined = " ".join(words)\n    if len(joined) <= width:\n        return joined\n    kept, length = [], 0\n    for word in words:\n        extra = len(word) + (1 if kept else 0)\n        if length + extra + len(placeholder) >= width:\n            break\n        kept.append(word)\n        length += extra\n    if not kept:\n        raise ValueError("placeholder too large for max width")\n    return " ".join(kept) + placeholder\n')
with open('history.py', 'w', encoding='utf-8') as f:
    f.write('GOOD = open("wraptext.py").read().replace("length + extra + len(placeholder) >= width", "length + extra + len(placeholder) > width")\nBAD = open("wraptext.py").read()\n\nCOMMITS = [\n    ("a1f3c20", "Add shorten()", GOOD),\n    ("b27e9d4", "Document shorten()", GOOD),\n    ("c3d81aa", "Collapse whitespace first", GOOD),\n    ("d4b0e6f", "Count the placeholder once", BAD),\n    ("e58c7b1", "Tidy names", BAD),\n    ("f6a2d93", "Add a module docstring", BAD),\n    ("g7e4410", "Raise ValueError for tiny widths", BAD),\n    ("h8c95fe", "Speed up the loop", BAD),\n]\n\n\ndef load(source):\n    namespace = {}\n    exec(source, namespace)\n    return namespace["shorten"]\n')

**Predict before you run:** what will this print?



In [ ]:
placeholder, width = " [...]", 18
length = 0
for word in ["Hello", "world,"]:
    extra = len(word) + (1 if length else 0)
    print(word, length + extra + len(placeholder), length + extra + len(placeholder) >= width)
    length += extra

> **Key idea.** Find the line where the decision goes wrong and explain the cause in one sentence; read the commit that introduced it. Make the smallest change that fixes the cause, nothing else. Compare old and new behaviour on many inputs, edges included: every difference should be a correction. Check properties every right answer has (fits, is a prefix, keeps as many words as fit).

### Exercise: Tracing the decisions

Write `trace_loop(text, width, placeholder=" [...]")`, following the buggy loop (`>=`): for each word in turn, the line `"WORD: TOTAL keep"` or `"WORD: TOTAL stop"`, where TOTAL is the length so far plus the word (and a space if any word is kept) plus the placeholder. Stop after the first `stop`. If the whole collapsed text fits the width, return `["fits: TEXT"]`.

In [ ]:
def trace_loop(text, width, placeholder=" [...]"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_trace():\n    \"where it goes wrong\"\n    assert trace_loop(\"Hello world, nice day\", 18) == [\"Hello: 11 keep\", \"world,: 18 stop\"]\n    assert trace_loop(\"Hello a nice day today\", 18) == [\"Hello: 11 keep\", \"a: 13 keep\", \"nice: 18 stop\"]\n    assert trace_loop(\"Hello   world\", 11) == [\"fits: Hello world\"]")

### Exercise: The fix

Write the corrected `shorten(text, width, placeholder=" [...]")`: collapse whitespace; if the result fits the width, return it; otherwise keep as many words from the start as fit **with** the placeholder (text that exactly fills the width fits) and add the placeholder; if not even one word fits, raise `ValueError("placeholder too large for max width")`.

In [ ]:
def shorten(text, width, placeholder=" [...]"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reported():\n    \"the reported case\"\n    assert shorten(\"Hello world, nice day\", 18) == \"Hello world, [...]\"\n    assert shorten(\"Hello a nice day today\", 18) == \"Hello a nice [...]\"\n\ndef test_edges():\n    \"one character either side\"\n    assert shorten(\"Hello world, nice day\", 17) == \"Hello [...]\"\n    assert shorten(\"Hello world, nice day\", 21) == \"Hello world, nice day\"\n    assert shorten(\"Hello   world\", 11) == \"Hello world\"\n    assert shorten(\"a b c d e f\", 9) == \"a b [...]\"\n    assert shorten(\"Hello world\", 8, placeholder=\"..\") == \"Hello..\"\n    try:\n        shorten(\"one two\", 3)\n    except ValueError as e:\n        assert str(e) == \"placeholder too large for max width\"\n    else:\n        raise AssertionError(\"tiny width\")")

### Exercise: Measuring the diff

Write `diff_stats(old, new, name="wraptext.py")`: compare the two sources line by line with `difflib.unified_diff` (from `a/NAME` to `b/NAME`, `lineterm=""`) and return `(added, removed, text)`: the counts of lines starting with `+` and `-` (not counting the `+++` and `---` headers) and the diff joined by newlines.

In [ ]:
import difflib

def diff_stats(old, new, name="wraptext.py"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import GOOD, BAD\n\ndef test_fix():\n    \"one line\"\n    added, removed, text = diff_stats(BAD, GOOD)\n    assert (added, removed) == (1, 1)\n    lines = text.splitlines()\n    assert lines[:3] == [\"--- a/wraptext.py\", \"+++ b/wraptext.py\", \"@@ -14,7 +14,7 @@\"]\n    assert \"-        if length + extra + len(placeholder) >= width:\" in lines and \"+        if length + extra + len(placeholder) > width:\" in lines\n    assert diff_stats(\"a\\n\", \"a\\n\") == (0, 0, \"\")")

### Exercise: What else changed?

Write `outcome(fn, args)` returning `repr(fn(*args))`, or the exception's class name if it raises one. Then `behaviour_changes(old, new, inputs)`: for each `args` tuple in `inputs`, in order, where the outcomes differ, `(args, OLD, NEW)`.

In [ ]:
def outcome(fn, args):
    ___

def behaviour_changes(old, new, inputs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import GOOD, BAD, load\n\nINPUTS = [(\"Hello world, nice day\", 18), (\"Hello world, nice day\", 17), (\"Hello world, nice day\", 21), (\"Hello   world\", 11),\n          (\"Hello a nice day today\", 18), (\"Supercalifragilistic word\", 10), (\"a b c d e f\", 9), (\"one two\", 3)]\n\ndef test_changes():\n    \"only corrections\"\n    assert behaviour_changes(load(BAD), load(GOOD), INPUTS) == [\n        ((\"Hello world, nice day\", 18), \"'Hello [...]'\", \"'Hello world, [...]'\"),\n        ((\"Hello a nice day today\", 18), \"'Hello a [...]'\", \"'Hello a nice [...]'\"),\n        ((\"a b c d e f\", 9), \"'a [...]'\", \"'a b [...]'\"),\n    ]\n    assert outcome(load(GOOD), (\"one two\", 3)) == \"ValueError\"")

### Exercise: Properties of every right answer

Write `violations(fn, inputs, placeholder=" [...]")`: for each `(text, width)`, call `fn(text, width)` (skip inputs where it raises `ValueError`). Report `"TEXT, WIDTH: too long (N)"` if the result is longer than the width; otherwise, if it ends with the placeholder, `"TEXT, WIDTH: not a prefix"` if the words before it aren't the text's first words, or `"TEXT, WIDTH: could keep WORD"` if one more word (with the placeholder) would still fit; otherwise `"TEXT, WIDTH: changed the text"` unless it's the collapsed text. TEXT and WORD are written with `repr`.

In [ ]:
def violations(fn, inputs, placeholder=" [...]"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import GOOD, BAD, load\n\nINPUTS = [(\"Hello world, nice day\", 18), (\"Hello world, nice day\", 17), (\"Hello world, nice day\", 21), (\"Hello   world\", 11),\n          (\"Hello a nice day today\", 18), (\"Supercalifragilistic word\", 10), (\"a b c d e f\", 9), (\"one two\", 3)]\n\ndef test_buggy():\n    \"the bug breaks one property\"\n    assert violations(load(BAD), INPUTS) == [\"'Hello world, nice day', 18: could keep 'world,'\", \"'Hello a nice day today', 18: could keep 'nice'\",\n                                            \"'a b c d e f', 9: could keep 'b'\"]\n    assert violations(load(GOOD), INPUTS) == []\n\ndef test_others():\n    \"other kinds of wrong\"\n    assert violations(lambda t, w: t, [(\"Hello world, nice day\", 10)]) == [\"'Hello world, nice day', 10: too long (21)\"]\n    assert violations(lambda t, w: \"world, [...]\", [(\"Hello world, nice day\", 12)]) == [\"'Hello world, nice day', 12: not a prefix\"]\n    assert violations(lambda t, w: t.upper(), [(\"ab\", 5)]) == [\"'ab', 5: changed the text\"]")

### Bug hunt: A fix that overshoots

This "fix" allows one character too many, so at width 17 it keeps `world,` and returns 18 characters: it prints `18`. Make it the smallest correct fix, so it prints `11` (and still gives `Hello world, [...]` at width 18).

In [ ]:
def shorten(text, width, placeholder=" [...]"):
    words = text.split()
    joined = " ".join(words)
    if len(joined) <= width:
        return joined
    kept, length = [], 0
    for word in words:
        extra = len(word) + (1 if kept else 0)
        if length + extra + len(placeholder) > width + 1:
            break
        kept.append(word)
        length += extra
    if not kept:
        raise ValueError("placeholder too large for max width")
    return " ".join(kept) + placeholder

print(len(shorten("Hello world, nice day", 17)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"never longer than the width\"\n    expect_output(\"11\")\n    assert shorten(\"Hello world, nice day\", 18) == \"Hello world, [...]\"")

### Bug hunt: Spaces that weren't collapsed

The "does it fit already?" check measures the original text, with its extra spaces, so `"Hello   world"` at width 11 is shortened to `Hello [...]` although `Hello world` fits. Measure the collapsed text, so it prints `Hello world`.

In [ ]:
def shorten(text, width, placeholder=" [...]"):
    words = text.split()
    if len(text) <= width:
        return " ".join(words)
    kept, length = [], 0
    for word in words:
        extra = len(word) + (1 if kept else 0)
        if length + extra + len(placeholder) > width:
            break
        kept.append(word)
        length += extra
    return " ".join(kept) + placeholder

print(shorten("Hello   world", 11))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"collapse first\"\n    expect_output(\"Hello world\")")

### Puzzle: The corrected loop

Arrange the loop so text that exactly fills the width is kept. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the comparison operator the fix restores, as a string
q = ___   # len(shorten("Hello world, nice day", 17)) after the fix
r = ___   # lines added plus lines removed in the fix's diff
s = ___   # True or False: rename a confusing variable in the same pull request as the fix
t = ___   # the name of the library that generates inputs for property-based tests

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \">\" and q == 11 and r == 2, \"p, q and r\"\n    assert s is False and t == \"Hypothesis\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: reviewing a fix

Write `review_fix(old_source, new_source, inputs)` with today's helpers (given) and `load` from `history.py`. Return `(lines, verdict)`:

1. `"diff: +A -R"` from `diff_stats`;
2. `"changed: N inputs"`, then for each change `"  TEXT, WIDTH: OLD becomes NEW"` (TEXT with `repr`);
3. `"violations: before B, after V"` from `violations` on the old and new functions.

`verdict` is `"good fix"` if A + R is at most 4, there are no violations after, and every changed input had a violation before; otherwise `"rethink: "` and the failing reasons in this order, joined by `", "`: `diff too large`, `still wrong`, `changes correct behaviour`.

In [ ]:
import difflib
from history import load

def diff_stats(old, new, name="wraptext.py"):
    lines = list(difflib.unified_diff(old.splitlines(), new.splitlines(), f"a/{name}", f"b/{name}", lineterm=""))
    added = sum(1 for l in lines if l.startswith("+") and not l.startswith("+++"))
    removed = sum(1 for l in lines if l.startswith("-") and not l.startswith("---"))
    return added, removed, "\n".join(lines)

def outcome(fn, args):
    try:
        return repr(fn(*args))
    except Exception as e:
        return type(e).__name__

def behaviour_changes(old, new, inputs):
    return [(args, outcome(old, args), outcome(new, args)) for args in inputs if outcome(old, args) != outcome(new, args)]

def violations(fn, inputs, placeholder=" [...]"):
    out = []
    for text, width in inputs:
        try:
            result = fn(text, width)
        except ValueError:
            continue
        words = text.split()
        if len(result) > width:
            out.append(f"{text!r}, {width}: too long ({len(result)})")
        elif result.endswith(placeholder):
            kept = result[:-len(placeholder)].split()
            if kept != words[:len(kept)]:
                out.append(f"{text!r}, {width}: not a prefix")
            elif len(kept) < len(words) and len(" ".join(words[:len(kept) + 1]) + placeholder) <= width:
                out.append(f"{text!r}, {width}: could keep {words[len(kept)]!r}")
        elif result != " ".join(words):
            out.append(f"{text!r}, {width}: changed the text")
    return out

def review_fix(old_source, new_source, inputs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from history import GOOD, BAD\n\nINPUTS = [(\"Hello world, nice day\", 18), (\"Hello world, nice day\", 17), (\"Hello world, nice day\", 21), (\"Hello   world\", 11),\n          (\"Hello a nice day today\", 18), (\"Supercalifragilistic word\", 10), (\"a b c d e f\", 9), (\"one two\", 3)]\n\ndef test_good():\n    \"the one-character fix\"\n    lines, verdict = review_fix(BAD, GOOD, INPUTS)\n    assert lines == [\"diff: +1 -1\", \"changed: 3 inputs\",\n                     \"  'Hello world, nice day', 18: 'Hello [...]' becomes 'Hello world, [...]'\",\n                     \"  'Hello a nice day today', 18: 'Hello a [...]' becomes 'Hello a nice [...]'\",\n                     \"  'a b c d e f', 9: 'a [...]' becomes 'a b [...]'\",\n                     \"violations: before 3, after 0\"]\n    assert verdict == \"good fix\"\n\ndef test_overreach():\n    \"a fix that also changes the error\"\n    new = GOOD.replace('raise ValueError(\"placeholder too large for max width\")', 'return placeholder.strip()[:width]')\n    new = new.replace(\"kept, length = [], 0\", \"kept = []\\n    length = 0\")\n    lines, verdict = review_fix(BAD, new, INPUTS)\n    assert lines[0] == \"diff: +4 -3\" and lines[1] == \"changed: 5 inputs\" and lines[-1] == \"violations: before 3, after 2\"\n    assert lines[-2] == \"  'one two', 3: ValueError becomes '[..'\"\n    assert verdict == \"rethink: diff too large, still wrong, changes correct behaviour\"\n    assert review_fix(BAD, BAD, INPUTS) == ([\"diff: +0 -0\", \"changed: 0 inputs\", \"violations: before 3, after 3\"], \"rethink: still wrong\")")

## Warm-up

With `world,` the total would be exactly 18, and `>=` treats "exactly the width" as "too long". That's the cause, in one sentence: **the loop stops when the text would exactly fill the width, but text that exactly fills the width fits.** Commit `d4b0e6f` changed `>` to `>=`.

## Understanding the cause

Find the line where the program's decision goes wrong, and say why, before editing anything. Print (or trace) the values at each step, as above; read the commit that introduced the bug (`git show d4b0e6f`) to see what its author intended (here, "count the placeholder once", a tidy-up that changed a comparison by accident). If you can't explain the cause, you can't be sure your change fixes it rather than hiding it.

## The smallest change

The fix here is one character: `>=` back to `>`. Resist everything else you noticed on the way (a variable name you'd choose differently, a loop you'd write with `itertools`): each extra change makes the review longer, risks a new bug, and mixes two decisions in one pull request. If something else needs fixing, it's another issue and another pull request. The diff for a good bug fix is often a few lines plus its test.

## Nothing else changes

A fix should change behaviour **only** where the old behaviour was wrong. Check it: run the old and new versions on many inputs (the reported case, the edges such as an exact fit, one character either side, a single very long word, a width too small for the placeholder, extra whitespace) and look at every input where they differ. Each difference should be a correction. Any other difference is a change in behaviour someone may depend on.

## Properties of a right answer

Some facts are true of every correct result, whatever the input: it's never longer than the width; if it ends with the placeholder, the words before it are the **first** words of the text, in order; and it keeps **as many words as fit** (adding the next one would overflow). Checking these on many inputs is **property-based testing** (Hypothesis does it automatically on your own computer, generating the inputs too). Here the third property is exactly the one the bug breaks.

## Practice

## Quiz

## Daily challenge

## Your turn

1. **Explain your bug's cause** in one sentence, written in your notes, with the line where the decision goes wrong.
2. **Make the smallest fix** on a branch (`git switch -c fix-gh-NNNNN`), and look at `git diff`: is every changed line needed?
3. **Compare old and new** on the reported case and at least ten edge cases, and check any properties a right answer must have.
4. **Commit** with a message saying what changed and why (Day 4 has more on that).

## Before you go

Tomorrow you'll write the test that proves the fix and keeps the bug from ever coming back.